E5 FLOODING MODELING

In [2]:
from google.colab import drive
import os

# Montar el drive
drive.mount('/content/drive')

Mounted at /content/drive


Start and Preprocessing for Dynbamiuc Predictive Model

In [4]:
import os
import json
import numpy as np
import rasterio
import time

# 1. Rutas Absolutas (Asegúrate de que el Drive esté montado)
base_path_e4 = "/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E4"
base_path_e5 = "/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E5"

# Crear estructura de carpetas con manejo de errores
subcarpetas = [
    "Datasets/Train",
    "Datasets/Metadata",
    "Models/Weights",
    "Results/Predictions"
]

def crear_carpetas():
    for folder in subcarpetas:
        p = os.path.join(base_path_e5, folder)
        if not os.path.exists(p):
            try:
                os.makedirs(p, exist_ok=True)
                print(f"Carpeta creada: {folder}")
            except Exception as e:
                print(f"Error al crear {folder}: {e}")

# Función para guardar archivos .npy con reintentos (Evita el Errno 107)
def safe_save_npy(path, data):
    for i in range(3): # 3 Intentos de guardado
        try:
            np.save(path, data)
            print(f"Guardado exitoso: {os.path.basename(path)}")
            return
        except OSError:
            print(f"Fallo de conexión en Drive. Reintentando {i+1}/3...")
            time.sleep(5) # Espera 5 segundos para que Drive reconecte
    print(f"CRÍTICO: No se pudo guardar {path}")

# 2. Procesamiento de Datos (Sugerencias Aplicadas)
def inicializar_e5():
    crear_carpetas()

    path_cem = os.path.join(base_path_e4, "Input data/2. Mapa Veracruz georeferenciado.tif")
    gt_path_e4 = os.path.join(base_path_e4, "PyTorch Metadata/Ground_Truth_Tensor.npy")

    # A. Normalización del CEM
    if os.path.exists(path_cem):
        with rasterio.open(path_cem) as src:
            cem = src.read(1).astype(np.float32)
            cem[cem < -900] = np.min(cem[cem > -900])
            cem_norm = (cem - cem.min()) / (cem.max() - cem.min())
            safe_save_npy(os.path.join(base_path_e5, "Datasets/Train/CEM_Normalized.npy"), cem_norm)

    # B. Data Augmentation (Espejeo)
    if os.path.exists(gt_path_e4):
        gt = np.load(gt_path_e4)
        safe_save_npy(os.path.join(base_path_e5, "Datasets/Train/GT_Original.npy"), gt)
        # Aumentar Dataset
        safe_save_npy(os.path.join(base_path_e5, "Datasets/Train/GT_FlipH.npy"), np.flip(gt, axis=1))
        safe_save_npy(os.path.join(base_path_e5, "Datasets/Train/GT_FlipV.npy"), np.flip(gt, axis=0))

    # C. Enriquecimiento de Metadata
    meta_e4 = os.path.join(base_path_e4, "PyTorch Metadata/Metadata_Dataset.json")
    if os.path.exists(meta_e4):
        with open(meta_e4, 'r') as f:
            data = json.load(f)

        data["ia_config"] = {
            "status": "Ready for E5",
            "normalization_range": [0, 1],
            "augmentation": "Implemented"
        }

        with open(os.path.join(base_path_e5, "Datasets/Metadata/Enriched_Metadata.json"), 'w') as f:
            json.dump(data, f, indent=4)

if __name__ == "__main__":
    inicializar_e5()

Guardado exitoso: CEM_Normalized.npy
Guardado exitoso: GT_Original.npy
Guardado exitoso: GT_FlipH.npy
Guardado exitoso: GT_FlipV.npy
